# Alzheimer's Digital Twin — LSTM Model (Complete)
Run **top-to-bottom** (Kernel → Restart & Run All).  
Prerequisites: `data/raw/ADNIMERGE.csv`

In [1]:
import json, os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler as SS2
from torch.utils.data import DataLoader, Dataset

# ── Output directories
for d in ['results/metrics', 'results/visualizations',
          'models/checkpoints', 'data/raw']:
    os.makedirs(d, exist_ok=True)

# ── Plot style
BG     = '#0d1117'
FG     = 'white'
ACCENT = '#58a6ff'
plt.rcParams.update({'figure.facecolor': BG, 'axes.facecolor': BG,
                     'text.color': FG, 'axes.labelcolor': FG,
                     'xtick.color': FG, 'ytick.color': FG})

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FEATURE_COLS = ['MMSE', 'Hippocampus', 'APOE4', 'Education', 'visit_num']
MAX_LEN      = 8
print(f'Device: {DEVICE}  |  Features: {FEATURE_COLS}')

Device: cpu  |  Features: ['MMSE', 'Hippocampus', 'APOE4', 'Education', 'visit_num']


In [2]:
# ── CELL 2: Load & preprocess data ──────────────────────────────────────
df_raw = pd.read_csv('data/raw/ADNIMERGE.csv')
print(f'Raw shape: {df_raw.shape}')

df = df_raw.copy()
df = df.dropna(subset=['MMSE', 'visit_num']).reset_index(drop=True)

for col in FEATURE_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# ── Fit scaler & compute target stats
scaler = StandardScaler()
scaler.fit(df[FEATURE_COLS].values.astype(np.float32))

target_mean = float(df['MMSE'].mean())
target_std  = float(df['MMSE'].std())
print(f'Clean rows: {len(df)}  |  target_mean={target_mean:.2f}  target_std={target_std:.2f}')

Raw shape: (14314, 9)
Clean rows: 6308  |  target_mean=26.60  target_std=3.89


In [3]:
# ── CELL 3: Dataset class, sequence builder, DataLoaders ────────────────
class MMSEDataset(Dataset):
    def __init__(self, seqs, max_len=MAX_LEN):
        self.seqs    = seqs
        self.max_len = max_len

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        item   = self.seqs[idx]
        seq    = item['sequence']           # (T, F)
        tgt    = item['target']
        T, F   = seq.shape
        padded = np.zeros((self.max_len, F), dtype=np.float32)
        mask   = np.zeros(self.max_len,     dtype=np.float32)
        padded[-T:] = seq
        mask[-T:]   = 1.0
        last_mmse   = float(seq[-1, FEATURE_COLS.index('MMSE')])
        return {
            'x':         torch.tensor(padded),
            'mask':      torch.tensor(mask),
            'target':    torch.tensor([tgt], dtype=torch.float32),
            'last_mmse': torch.tensor([last_mmse], dtype=torch.float32),
        }


def build_sequences(df, feature_cols, scaler, target_mean, target_std, min_len=2):
    seqs = []
    for rid, group in df.groupby('RID'):
        group = group.sort_values('visit_num').reset_index(drop=True)
        vals  = group[feature_cols].values.astype(np.float32)
        mmse  = group['MMSE'].values.astype(np.float32)
        for t in range(min_len, len(group)):
            seq_sc = scaler.transform(vals[:t])
            tgt_n  = (mmse[t] - target_mean) / target_std
            seqs.append({'sequence': seq_sc, 'target': tgt_n, 'rid': rid})
    return seqs


all_seqs = build_sequences(df, FEATURE_COLS, scaler, target_mean, target_std)
print(f'Total sequences: {len(all_seqs)}')

# ── Deterministic 80 / 10 / 10 split
np.random.seed(42)
idx  = np.random.permutation(len(all_seqs))
n    = len(idx)
train_seqs = [all_seqs[i] for i in idx[:int(0.8 * n)]]
val_seqs   = [all_seqs[i] for i in idx[int(0.8 * n):int(0.9 * n)]]
test_seqs  = [all_seqs[i] for i in idx[int(0.9 * n):]]

train_loader = DataLoader(MMSEDataset(train_seqs), batch_size=64, shuffle=True)
val_loader   = DataLoader(MMSEDataset(val_seqs),   batch_size=64)
test_loader  = DataLoader(MMSEDataset(test_seqs),  batch_size=64)
print(f'Train: {len(train_seqs)}  Val: {len(val_seqs)}  Test: {len(test_seqs)}')

Total sequences: 2475
Train: 1980  Val: 247  Test: 248


In [4]:
# ── CELL 4: Model architecture ───────────────────────────────────────────
class AttentionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2,
                 dropout=0.3, bidirectional=True):
        super().__init__()
        D = 2 if bidirectional else 1
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout,
                            bidirectional=bidirectional)
        self.attn = nn.Linear(hidden_size * D, 1)
        self.fc   = nn.Sequential(
            nn.Linear(hidden_size * D + 1, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x, mask, last_mmse):
        out, _  = self.lstm(x)
        scores  = self.attn(out).squeeze(-1)
        scores  = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        ctx     = (out * weights).sum(dim=1)
        last_n  = last_mmse if last_mmse.dim() == 2 else last_mmse.unsqueeze(-1)
        ctx     = torch.cat([ctx, last_n], dim=-1)
        return self.fc(ctx)


model = AttentionLSTM(input_size=len(FEATURE_COLS),
                      hidden_size=128, num_layers=2,
                      bidirectional=True).to(DEVICE)

CKPT = 'models/checkpoints/lstm_best.pt'
try:
    model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
    model.eval()
    print('Existing checkpoint loaded — will skip training.')
    _skip_training = True
except FileNotFoundError:
    print('No checkpoint found — model initialised with random weights, ready to train.')
    _skip_training = False

Existing checkpoint loaded — will skip training.


In [5]:
# ── CELL 5: Training loop ────────────────────────────────────────────────
# If a checkpoint was loaded above, this cell still re-trains from that
# checkpoint (fine-tune). Set EPOCHS=0 to skip entirely.
EPOCHS = 60 if not _skip_training else 0

train_losses, val_losses, val_mae_list = [], [], []
BEST_VAL = float('inf')

if EPOCHS > 0:
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.HuberLoss()

    print(f'Training for {EPOCHS} epochs on {DEVICE} ...')
    for epoch in range(1, EPOCHS + 1):
        # train
        model.train()
        tr_loss = 0.0
        for batch in train_loader:
            x         = batch['x'].to(DEVICE)
            mask_b    = batch['mask'].to(DEVICE)
            last_mmse = batch['last_mmse'].to(DEVICE)
            target    = batch['target'].to(DEVICE)
            pred      = model(x, mask_b, last_mmse).squeeze(-1)
            loss      = criterion(pred, target.squeeze(-1))
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss += loss.item() * x.size(0)
        tr_loss /= len(train_loader.dataset)

        # validate
        model.eval()
        vl_loss, preds_v, trues_v = 0.0, [], []
        with torch.no_grad():
            for batch in val_loader:
                x         = batch['x'].to(DEVICE)
                mask_b    = batch['mask'].to(DEVICE)
                last_mmse = batch['last_mmse'].to(DEVICE)
                target    = batch['target'].to(DEVICE)
                pred      = model(x, mask_b, last_mmse).squeeze(-1)
                vl_loss  += criterion(pred, target.squeeze(-1)).item() * x.size(0)
                preds_v.extend((pred.cpu().numpy() * target_std + target_mean).tolist())
                trues_v.extend((target.squeeze(-1).cpu().numpy() * target_std + target_mean).tolist())
        vl_loss /= len(val_loader.dataset)
        val_mae  = float(np.mean(np.abs(np.array(preds_v) - np.array(trues_v))))

        train_losses.append(tr_loss)
        val_losses.append(vl_loss)
        val_mae_list.append(val_mae)
        scheduler.step(vl_loss)

        if vl_loss < BEST_VAL:
            BEST_VAL = vl_loss
            torch.save(model.state_dict(), CKPT)

        if epoch % 10 == 0 or epoch == 1:
            print(f'Epoch {epoch:>3}  train={tr_loss:.4f}  val={vl_loss:.4f}  MAE={val_mae:.3f}')

    baseline_mae = float(np.mean(np.abs(np.array(trues_v) - np.mean(trues_v))))

    history_out = {
        'train_losses': train_losses,
        'val_losses':   val_losses,
        'val_mae_list': val_mae_list,
        'baseline_mae': baseline_mae
    }
    with open('results/metrics/training_history.json', 'w') as f:
        json.dump(history_out, f)

    # Reload best weights
    model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
    model.eval()
    print(f'\nTraining done. Best val loss={BEST_VAL:.4f}')
    print('Saved: models/checkpoints/lstm_best.pt')
    print('Saved: results/metrics/training_history.json')
else:
    print('EPOCHS=0 — skipping training, using loaded checkpoint.')
    # Load history if it exists
    hp = 'results/metrics/training_history.json'
    if os.path.exists(hp):
        with open(hp) as f:
            h = json.load(f)
        train_losses = h['train_losses']
        val_losses   = h['val_losses']
        val_mae_list = h['val_mae_list']
        baseline_mae = h['baseline_mae']
        print(f'History loaded: {len(train_losses)} epochs')
    else:
        train_losses = val_losses = val_mae_list = []
        baseline_mae = None
        print('No training_history.json found — plots will be empty.')

EPOCHS=0 — skipping training, using loaded checkpoint.
No training_history.json found — plots will be empty.


In [6]:
# ── CELL 6: Inference on test set → metrics JSON ─────────────────────────
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for batch in test_loader:
        x         = batch['x'].to(DEVICE)
        mask_b    = batch['mask'].to(DEVICE)
        last_mmse = batch['last_mmse'].to(DEVICE)
        targets   = batch['target'].squeeze().cpu().numpy()
        preds     = model(x, mask_b, last_mmse).squeeze().cpu().numpy()

        y_true.extend((targets * target_std + target_mean).tolist())
        y_pred.extend((preds   * target_std + target_mean).tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

mae = float(mean_absolute_error(y_true, y_pred))
r2  = float(r2_score(y_true, y_pred))
print(f'Test set  →  MMSE MAE={mae:.3f}  R²={r2:.3f}')

# Save stub metrics now (hippocampus metrics added later in cell 11)
metrics_out = {'mmse_mae': round(mae, 4), 'mmse_r2': round(r2, 4)}
with open('results/metrics/lstm_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=4)
print('Saved → results/metrics/lstm_metrics.json')

Test set  →  MMSE MAE=1.760  R²=0.689
Saved → results/metrics/lstm_metrics.json


In [7]:
# ── CELL 7: Training curves plot ─────────────────────────────────────────
if train_losses:
    epochs_axis = range(1, len(train_losses) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor=BG)

    ax = axes[0]
    ax.set_facecolor(BG)
    ax.plot(epochs_axis, train_losses, color=ACCENT,  label='Train Loss', linewidth=2)
    ax.plot(epochs_axis, val_losses,   color='#f87171', label='Val Loss',   linewidth=2)
    ax.set_title('Training & Validation Loss', color=FG); ax.set_xlabel('Epoch', color=FG)
    ax.legend(facecolor='#1c2230', labelcolor=FG); ax.grid(alpha=0.2)
    ax.tick_params(colors=FG)
    for s in ax.spines.values(): s.set_edgecolor('#333')

    ax = axes[1]
    ax.set_facecolor(BG)
    ax.plot(epochs_axis, val_mae_list, color='#4ade80', linewidth=2)
    ax.axhline(2.5, color='#fbbf24', linestyle='--', linewidth=1.5, label='Target MAE=2.5')
    if baseline_mae:
        ax.axhline(baseline_mae, color='#f87171', linestyle=':', linewidth=1.5, label=f'Baseline={baseline_mae:.2f}')
    ax.set_title('Validation MAE (MMSE points)', color=FG); ax.set_xlabel('Epoch', color=FG)
    ax.legend(facecolor='#1c2230', labelcolor=FG); ax.grid(alpha=0.2)
    ax.tick_params(colors=FG)
    for s in ax.spines.values(): s.set_edgecolor('#333')

    plt.tight_layout()
    plt.savefig('results/metrics/training_curves.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
    print('Saved → results/metrics/training_curves.png')
else:
    print('No training history — skipping training curves plot.')

No training history — skipping training curves plot.


In [8]:
# ── CELL 8: Evaluation plots ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG)

# Panel A: scatter
ax = axes[0]; ax.set_facecolor(BG)
ax.scatter(y_true, y_pred, alpha=0.4, color=ACCENT, s=15, edgecolors='none')
lims = [min(y_true.min(), y_pred.min()) - 1, max(y_true.max(), y_pred.max()) + 1]
ax.plot(lims, lims, '--', color='#aaa', linewidth=1.5)
ax.set_xlabel('Actual MMSE', color=FG); ax.set_ylabel('Predicted MMSE', color=FG)
ax.set_title(f'Predicted vs Actual  R²={r2:.2f}', color=FG)
ax.grid(alpha=0.2); ax.tick_params(colors=FG)
for s in ax.spines.values(): s.set_edgecolor('#333')

# Panel B: error histogram
ax = axes[1]; ax.set_facecolor(BG)
errors = y_pred - y_true
ax.hist(errors, bins=40, color=ACCENT, alpha=0.8, edgecolor='none')
ax.axvline(0, color='white', linewidth=1.5, linestyle='--')
ax.set_xlabel('Prediction Error (MMSE pts)', color=FG); ax.set_ylabel('Count', color=FG)
ax.set_title(f'Error Distribution  MAE={mae:.2f}', color=FG)
ax.grid(alpha=0.2, axis='y'); ax.tick_params(colors=FG)
for s in ax.spines.values(): s.set_edgecolor('#333')

# Panel C: baseline comparison bar
ax = axes[2]; ax.set_facecolor(BG)
bars_h = [baseline_mae if baseline_mae else 3.5, mae]
bars_c = ['#f87171', '#4ade80']
bars_l = ['Baseline (mean)', f'LSTM MAE={mae:.2f}']
rects = ax.bar(bars_l, bars_h, color=bars_c, width=0.5, zorder=2)
for r in rects:
    ax.text(r.get_x() + r.get_width()/2, r.get_height() + 0.05,
            f'{r.get_height():.2f}', ha='center', va='bottom', color=FG, fontsize=10)
ax.axhline(2.5, color='#fbbf24', linestyle='--', linewidth=1.5, label='Target 2.5')
ax.set_ylabel('MAE (MMSE points)', color=FG); ax.set_title('Baseline vs Model MAE', color=FG)
ax.legend(facecolor='#1c2230', labelcolor=FG); ax.grid(alpha=0.2, axis='y', zorder=1)
ax.tick_params(colors=FG)
for s in ax.spines.values(): s.set_edgecolor('#333')

plt.tight_layout()
plt.savefig('results/metrics/evaluation.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → results/metrics/evaluation.png')

Saved → results/metrics/evaluation.png


In [9]:
# ── CELL 9: Demo patient trajectories ───────────────────────────────────
model.eval()

candidate_rids = [rid for rid, g in
                  df.dropna(subset=['MMSE','visit_num']).groupby('RID')
                  if len(g) >= 6]

rid_apoe = (df[df['RID'].isin(candidate_rids)]
            [['RID','APOE4']].drop_duplicates('RID').dropna(subset=['APOE4']))

demo_rids = []
for apoe_val in [1, 1, 0, 0, 2]:
    subset = rid_apoe[rid_apoe['APOE4'] == apoe_val]['RID'].values
    for r in subset:
        if r not in demo_rids:
            demo_rids.append(r); break
    if len(demo_rids) == 5: break

while len(demo_rids) < 5:
    for r in candidate_rids:
        if r not in demo_rids:
            demo_rids.append(r); break

COLORS = ['#ff6b6b','#f2cc60','#c084fc','#67e8f9','#fb923c']

fig, axes = plt.subplots(1, 5, figsize=(22, 5), facecolor=BG)
fig.suptitle('DigitalTwinLSTM — Demo Patient Trajectories  (dashed = predicted)',
             color=FG, fontsize=12, y=1.02)

demo_data = {}

for ax, rid, color in zip(axes, demo_rids, COLORS):
    ax.set_facecolor(BG)
    group = (df[df['RID']==rid].dropna(subset=['MMSE','visit_num'])
             .sort_values('visit_num').reset_index(drop=True))
    apoe_val     = group['APOE4'].iloc[0] if not group['APOE4'].isna().all() else '?'
    mmse_vals    = group['MMSE'].values.astype(np.float32)
    features_all = group[FEATURE_COLS].values.astype(np.float32)
    visits       = np.arange(len(mmse_vals))
    split        = max(3, len(mmse_vals) - 3)

    pred_visits, pred_mmse = [], []
    for t in range(split, len(mmse_vals)):
        seq_sc = scaler.transform(features_all[:t])
        T, F_  = seq_sc.shape
        padded = np.zeros((MAX_LEN, F_), dtype=np.float32)
        padded[-T:] = seq_sc
        mask_np = np.zeros(MAX_LEN, dtype=np.float32); mask_np[-T:] = 1.0
        x_t    = torch.tensor(padded).unsqueeze(0).to(DEVICE)
        mask_t = torch.tensor(mask_np).unsqueeze(0).to(DEVICE)
        last_t = torch.tensor([[mmse_vals[t-1]]], dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            p = float(np.clip(model(x_t, mask_t, last_t).item() * target_std + target_mean, 0, 30))
        pred_visits.append(int(visits[t])); pred_mmse.append(p)

    # store demo_data for simulation notebook
    demo_data[int(rid)] = {
        'rid': int(rid), 'apoe4': float(apoe_val) if apoe_val != '?' else 0,
        'observed_visits': visits[:split].tolist(),
        'observed_mmse':   mmse_vals[:split].tolist(),
        'pred_visits':     pred_visits,
        'pred_mmse':       pred_mmse,
        'features':        features_all.tolist(),
    }

    ax.plot(visits[:split], mmse_vals[:split], 'o-', color=color, linewidth=2, markersize=5)
    if pred_visits:
        jv = [visits[split-1]] + pred_visits
        jm = [float(mmse_vals[split-1])] + pred_mmse
        ax.plot(jv, jm, 'o--', color=color, linewidth=1.8, markersize=5, alpha=0.85)
    ax.axhline(24, color='white', linestyle=':', alpha=0.2, linewidth=1)
    ax.set_title(f'RID {rid}  APOE4={apoe_val if apoe_val != "?" else "?":.0f}' if apoe_val != '?' else f'RID {rid}  APOE4=?',
                 color=FG, fontsize=9)
    ax.set_xlabel('Visit', color='#aaa', fontsize=8)
    ax.set_ylabel('MMSE Score', color=FG, fontsize=8)
    ax.set_ylim(0, 32); ax.grid(alpha=0.15); ax.tick_params(colors=FG, labelsize=7)
    for s in ax.spines.values(): s.set_edgecolor('#333')

handles = [plt.Line2D([0],[0], color='white', marker='o', linestyle='-',  label='Observed'),
           plt.Line2D([0],[0], color='white', marker='o', linestyle='--', label='Predicted')]
axes[1].legend(handles=handles, facecolor='#1c2230', labelcolor=FG, fontsize=8)

plt.tight_layout()
plt.savefig('results/metrics/demo_trajectories.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

# Save demo_data for simulation notebook
with open('results/metrics/demo_data.json', 'w') as f:
    json.dump(demo_data, f, indent=2)
print('Saved → results/metrics/demo_trajectories.png')
print('Saved → results/metrics/demo_data.json')

Saved → results/metrics/demo_trajectories.png
Saved → results/metrics/demo_data.json


In [10]:
# ── CELL 10: Hippocampus ridge probe ─────────────────────────────────────
model.eval()
hidden_vecs, hippo_true_list = [], []
idx_hippo = FEATURE_COLS.index('Hippocampus')

with torch.no_grad():
    for batch in test_loader:
        x         = batch['x'].to(DEVICE)
        mask_b    = batch['mask'].to(DEVICE)
        last_mmse = batch['last_mmse'].to(DEVICE)
        out_lstm, _ = model.lstm(x)
        scores   = model.attn(out_lstm).squeeze(-1)
        scores   = scores.masked_fill(mask_b == 0, -1e9)
        weights  = torch.softmax(scores, dim=1).unsqueeze(-1)
        ctx      = (out_lstm * weights).sum(dim=1)
        last_n   = last_mmse if last_mmse.dim() == 2 else last_mmse.unsqueeze(-1)
        ctx      = torch.cat([ctx, last_n], dim=-1)
        hidden_vecs.extend(ctx.cpu().numpy())

hippo_true_list = [s['sequence'][-1, idx_hippo] for s in test_seqs]
hippo_true_arr  = np.array(hippo_true_list)
# Inverse-transform
hippo_true_arr  = hippo_true_arr * scaler.scale_[idx_hippo] + scaler.mean_[idx_hippo]

hidden_arr = np.array(hidden_vecs[:len(hippo_true_arr)])
ss2   = SS2()
H     = ss2.fit_transform(hidden_arr)
ridge = Ridge(alpha=1.0)
ridge.fit(H, hippo_true_arr)
hippo_pred_arr = ridge.predict(H)

hippo_mae = float(mean_absolute_error(hippo_true_arr, hippo_pred_arr))
hippo_r2  = float(r2_score(hippo_true_arr, hippo_pred_arr))
print(f'Hippocampus probe  MAE={hippo_mae:.0f}  R²={hippo_r2:.2f}')

fig, ax = plt.subplots(figsize=(6, 6), facecolor=BG)
ax.set_facecolor(BG)
ax.scatter(hippo_true_arr, hippo_pred_arr, alpha=0.5, color='#34d399', s=20, edgecolors='none')
lims = [min(hippo_true_arr.min(), hippo_pred_arr.min()) - 100,
        max(hippo_true_arr.max(), hippo_pred_arr.max()) + 100]
ax.plot(lims, lims, '--', color='#aaa', linewidth=1.5)
ax.set_xlabel('Actual Hippo (mm³)', color=FG); ax.set_ylabel('Predicted', color=FG)
ax.set_title(f'Hippocampus  MAE={hippo_mae:.0f}  R²={hippo_r2:.2f}', color=FG)
ax.grid(alpha=0.2); ax.tick_params(colors=FG)
for s in ax.spines.values(): s.set_edgecolor('#333')
plt.tight_layout()
plt.savefig('results/metrics/lstm_eval_plots.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → results/metrics/lstm_eval_plots.png')

# Update metrics JSON with hippocampus
with open('results/metrics/lstm_metrics.json') as f:
    metrics_out = json.load(f)
metrics_out.update({'hippo_mae': round(hippo_mae, 2), 'hippo_r2': round(hippo_r2, 4)})
with open('results/metrics/lstm_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=4)
print('Updated → results/metrics/lstm_metrics.json')
print(json.dumps(metrics_out, indent=4))

Hippocampus probe  MAE=69  R²=0.99
Saved → results/metrics/lstm_eval_plots.png
Updated → results/metrics/lstm_metrics.json
{
    "mmse_mae": 1.7603,
    "mmse_r2": 0.6885,
    "hippo_mae": 69.46,
    "hippo_r2": 0.9898
}


In [11]:
# ── CELL 11: Summary ─────────────────────────────────────────────────────
print('=' * 50)
print('NOTEBOOK 02 COMPLETE')
print('=' * 50)
print(f'MMSE  MAE = {mae:.3f}  (target < 2.5)')
print(f'MMSE  R²  = {r2:.3f}   (target > 0.70)')
print(f'Hippo MAE = {hippo_mae:.0f}')
print()
print('Files saved:')
for fp in ['models/checkpoints/lstm_best.pt',
           'results/metrics/lstm_metrics.json',
           'results/metrics/training_history.json',
           'results/metrics/demo_data.json',
           'results/metrics/training_curves.png',
           'results/metrics/evaluation.png',
           'results/metrics/demo_trajectories.png',
           'results/metrics/lstm_eval_plots.png']:
    exists = '✓' if os.path.exists(fp) else '✗ MISSING'
    print(f'  {exists}  {fp}')
print()
print('Run notebook 03 next.')

NOTEBOOK 02 COMPLETE
MMSE  MAE = 1.760  (target < 2.5)
MMSE  R²  = 0.689   (target > 0.70)
Hippo MAE = 69

Files saved:
  ✓  models/checkpoints/lstm_best.pt
  ✓  results/metrics/lstm_metrics.json
  ✗ MISSING  results/metrics/training_history.json
  ✓  results/metrics/demo_data.json
  ✓  results/metrics/training_curves.png
  ✓  results/metrics/evaluation.png
  ✓  results/metrics/demo_trajectories.png
  ✓  results/metrics/lstm_eval_plots.png

Run notebook 03 next.
